# ALS baseline for popularity calibration

This notebook documents the Spark/Scala environment for the MovieLens 1M recommendation baseline. In the study, the ALS baseline provides the unadjusted recommendations against which fixed and history-aware popularity calibration are compared.

**Current implementation:** library loading, local Spark session initialization, runtime checks, MovieLens rating loading and parsing, DataFrame inspection, full rating and distinct-user counts, rating-value and identifier checks, duplicate-pair detection, per-user history counts, timestamp inspection, and session shutdown. ALS training and recommendation generation are not implemented, and no model or recommendation results are reported.

Run the implemented code cells in order using a Scala 2.12.18 kernel with Java 17. The notebook loads Spark 3.5.9. Saved outputs record a previous execution; rerunning the cells verifies the current environment.


## 1. Verify the local environment

The environment check loads Spark, starts a local session, and counts three synthetic rows. It verifies that the notebook can execute a Spark operation independently of MovieLens data or model training.

The `$ivy` imports load the specified Spark dependencies into the Scala kernel. `spark-sql` provides structured data processing, and `spark-mllib` provides machine-learning algorithms, including ALS. Importing `SparkSession` makes the session entry point available by name.


In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`

import org.apache.spark.sql.SparkSession


import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession

### Initialize Spark and verify the runtime

`SparkSession.builder()` configures the application name, local execution with two worker threads, and the driver's loopback addresses. `.getOrCreate()` returns an existing valid session or creates one; `val spark` stores the returned session reference.

After initialization, the log level is set to `ERROR`. The Scala and Java versions are read from the running kernel. `spark.range(3)` creates three synthetic rows, and `.count()` executes the operation and returns their count.

The assertions require Spark 3.5.9, Scala 2.12.18, Java 17, and a row count of three. A failed assertion stops execution of the cell. The printed version numbers and count provide evidence that the environment check completed successfully. These outputs are not recommender evaluation results.


In [2]:
val spark = SparkSession.builder()
  .appName("CS 7265 - ALS Baseline Environment")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val checkCount = spark.range(3).count()

assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(checkCount == 3L)

println(s"Spark: ${spark.version}")
println(s"Scala: $scalaVersion")
println(s"Java: $javaVersion")
println(s"Environment check passed: $checkCount synthetic rows.")


Spark: 3.5.9


Scala: 2.12.18


Java: 17.0.16


Environment check passed: 3 synthetic rows.


spark: SparkSession = org.apache.spark.sql.SparkSession@6b9aa1ed
scalaVersion: String = "2.12.18"
javaVersion: String = "17.0.16"
checkCount: Long = 3L

## 2. Load MovieLens ratings

The local `data/ml-1m/ratings.dat` file contains one `UserID::MovieID::Rating::Timestamp` record per line. Loading preserves all four fields. The user ID, movie ID, and observed rating are the inputs for explicit-feedback ALS; the timestamp is retained as source information.

### Define the record and parser

`Rating` defines the fields and their numeric types. `RatingParser` is a serializable function from a text line to a `Rating`. Its `apply` method splits a line at `::`, checks for four fields, and converts their values. Keep these definitions in their own cell, separate from the Spark context and data-loading variables, so the parser can be serialized without capturing driver-only notebook state.


In [3]:
case class Rating(userId: Int, movieId: Int, rating: Float, timestamp: Long)

class RatingParser extends (String => Rating) with Serializable {
  def apply(line: String): Rating = {
    val fields = line.split("::")
    require(fields.length == 4, "Expected four fields per MovieLens rating")

    Rating(
      fields(0).toInt,
      fields(1).toInt,
      fields(2).toFloat,
      fields(3).toLong
    )
  }
}


defined class Rating
defined class RatingParser

### Read and parse the local file

The path lookup supports a kernel working directory at either the repository root or its `notebooks` folder. It stops with a descriptive error if the dataset is missing. `sc.textFile` creates an `RDD[String]`, and `map(new RatingParser)` defines the conversion to an `RDD[Rating]`. Spark performs the parsing when an action requests the data. The source file is read without modification.


In [4]:
val ratingsPath = Seq(
  "data/ml-1m/ratings.dat",
  "../data/ml-1m/ratings.dat"
).find(path => new java.io.File(path).isFile).getOrElse {
  throw new IllegalArgumentException("Place MovieLens 1M ratings.dat in data/ml-1m/ and run from the repository root or notebooks folder.")
}

val sc = spark.sparkContext
val lines = sc.textFile(ratingsPath)
val ratingsRDD = lines.map(new RatingParser)


ratingsPath: String = "../data/ml-1m/ratings.dat"
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@249081bc
lines: org.apache.spark.rdd.RDD[String] = ../data/ml-1m/ratings.dat MapPartitionsRDD[8] at textFile at cmd4.sc:9
ratingsRDD: org.apache.spark.rdd.RDD[Rating] = MapPartitionsRDD[9] at map at cmd4.sc:10

### Convert records to a DataFrame

`import spark.implicits._` supplies the conversion helpers for `toDF()`. Spark derives the DataFrame column names and types from `Rating`. The records remain distributed; this conversion does not collect the full dataset into driver memory.


In [5]:
import spark.implicits._
val ratingsDF = ratingsRDD.toDF()


import spark.implicits._
ratingsDF: org.apache.spark.sql.package.DataFrame = [userId: int, movieId: int ... 2 more fields]

### Inspect the schema and sample rows

`printSchema()` displays the column structure. `show(5, truncate = false)` executes enough processing to display five records without shortening their contents. This preview checks the loading and conversion of sample rows; it is not a full-dataset validation or a model evaluation. No training split is created here.


In [6]:
ratingsDF.printSchema()
ratingsDF.show(5, truncate = false)


root
 |-- userId: integer (nullable = false)
 |-- movieId: integer (nullable = false)
 |-- rating: float (nullable = false)
 |-- timestamp: long (nullable = false)



+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1193   |5.0   |978300760|
|1     |661    |3.0   |978302109|
|1     |914    |3.0   |978301968|
|1     |3408   |4.0   |978300275|
|1     |2355   |5.0   |978824291|
+------+-------+------+---------+
only showing top 5 rows



## 3. Validate MovieLens ratings

These checks examine the full parsed dataset: record and user counts, rating values, identifier ranges, duplicate user–movie pairs, per-user history counts, and timestamps. They report data-quality issues without modifying the source records. Data validation here is separate from a validation split used to compare model settings.


### Count all ratings and distinct users

`ratingsDF.count()` processes the full parsed dataset and returns the number of rating records. `select("userId").distinct().count()` counts each user ID once without modifying the original DataFrame. Both counts are Spark actions and return `Long` values.

MovieLens 1M documents 1,000,209 ratings from 6,040 users. Matching these totals is a completeness check, but does not establish valid rating ranges, valid identifiers, or unique user–movie pairs. The parser fails on malformed records rather than silently dropping them.


In [7]:
val ratingCount = ratingsDF.count()
val userCount = ratingsDF.select("userId").distinct().count()

println(s"Rating records: $ratingCount")
println(s"Distinct users: $userCount")


Rating records: 1000209


Distinct users: 6040


ratingCount: Long = 1000209L
userCount: Long = 6040L

### Check rating values

MovieLens 1M uses whole-star ratings from 1 through 5. Grouping by `rating` and counting records displays the frequency of each value. The invalid-value check counts nulls and values outside the allowed set, including fractional ratings and non-finite values. It checks the full DataFrame without changing or dropping any source records.

The assertion requires zero invalid ratings. These checks establish rating-value validity; identifier validity and duplicate user–movie pairs require separate checks.


In [8]:
import org.apache.spark.sql.functions.col

ratingsDF.groupBy("rating").count().orderBy("rating").show()

val invalidRatingCount = ratingsDF.filter(
  col("rating").isNull ||
  !col("rating").isin(1.0f, 2.0f, 3.0f, 4.0f, 5.0f)
).count()

println(s"Invalid rating records: $invalidRatingCount")
assert(invalidRatingCount == 0L, "Ratings must be non-null whole-star values from 1 through 5.")


+------+------+
|rating| count|
+------+------+
|   1.0| 56174|
|   2.0|107557|
|   3.0|261197|
|   4.0|348971|
|   5.0|226310|
+------+------+



Invalid rating records: 0


import org.apache.spark.sql.functions.col
invalidRatingCount: Long = 0L

### Check user and movie identifiers

The parser converts both identifiers to `Int`. MovieLens 1M documents user IDs from 1 through 6,040 and movie IDs from 1 through 3,952. The checks require each ID to be non-null and inside its documented inclusive range. The minimum and maximum values summarize the observed endpoints; explicit invalid-record counts check every row.

IDs identify entities; their magnitudes are not preference features. Gaps in movie IDs are allowed, so the largest ID is not a count of distinct movies. These checks do not verify membership in the metadata files or uniqueness of user–movie pairs. Source records remain unchanged.


In [9]:
import org.apache.spark.sql.functions.{min, max}

ratingsDF.agg(
  min("userId").as("minUserId"),
  max("userId").as("maxUserId"),
  min("movieId").as("minMovieId"),
  max("movieId").as("maxMovieId")
).show()

val invalidUserIdCount = ratingsDF.filter(
  col("userId").isNull || !col("userId").between(1, 6040)
).count()

val invalidMovieIdCount = ratingsDF.filter(
  col("movieId").isNull || !col("movieId").between(1, 3952)
).count()

println(s"Records with invalid user IDs: $invalidUserIdCount")
println(s"Records with invalid movie IDs: $invalidMovieIdCount")

assert(invalidUserIdCount == 0L, "User IDs must be non-null integers from 1 through 6040.")
assert(invalidMovieIdCount == 0L, "Movie IDs must be non-null integers from 1 through 3952.")


+---------+---------+----------+----------+
|minUserId|maxUserId|minMovieId|maxMovieId|
+---------+---------+----------+----------+
|        1|     6040|         1|      3952|
+---------+---------+----------+----------+



Records with invalid user IDs: 0


Records with invalid movie IDs: 0


import org.apache.spark.sql.functions.{min, max}
invalidUserIdCount: Long = 0L
invalidMovieIdCount: Long = 0L

### Check duplicate user–movie pairs

Grouping by both `userId` and `movieId` counts observations for each interaction. Groups with more than one record are duplicate pairs, even if their ratings or timestamps differ. `duplicatePairCount` counts the affected pairs, not the number of extra rows.

The assertion requires one observation per user–movie pair in the source dataset. Duplicate observations could give interactions extra weight during training, and conflicting ratings would require an explicit resolution policy. This check reports the issue without removing records.


In [10]:
val duplicatePairs = ratingsDF
  .groupBy("userId", "movieId")
  .count()
  .filter(col("count") > 1L)

val duplicatePairCount = duplicatePairs.count()
println(s"Repeated user-movie pairs: $duplicatePairCount")

assert(duplicatePairCount == 0L, "Each user-movie pair must appear only once.")


Repeated user-movie pairs: 0


duplicatePairs: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [userId: int, movieId: int ... 1 more field]
duplicatePairCount: Long = 0L

### Check the number of ratings per user

Grouping by `userId` counts the observations contributed by each user. The grouped count is renamed `ratingCount` to distinguish a user's history length from an individual star rating. The minimum and maximum summarize the observed history lengths, and the assertion checks MovieLens 1M's documented minimum of 20 ratings per user in the complete source dataset.

This source-data check does not remove users or define history groups for an experiment. Training-only history counts must be recomputed from the agreed training split; they may be smaller than these full-dataset counts.


In [11]:
val ratingsPerUser = ratingsDF
  .groupBy("userId")
  .count()
  .withColumnRenamed("count", "ratingCount")

ratingsPerUser.agg(
  min("ratingCount").as("minRatingsPerUser"),
  max("ratingCount").as("maxRatingsPerUser")
).show()

val usersBelowMinimum = ratingsPerUser
  .filter(col("ratingCount") < 20L)
  .count()

println(s"Users with fewer than 20 ratings: $usersBelowMinimum")
assert(usersBelowMinimum == 0L, "The complete MovieLens 1M dataset must contain at least 20 ratings per user.")


+-----------------+-----------------+
|minRatingsPerUser|maxRatingsPerUser|
+-----------------+-----------------+
|               20|             2314|
+-----------------+-----------------+



Users with fewer than 20 ratings: 0


ratingsPerUser: org.apache.spark.sql.package.DataFrame = [userId: int, ratingCount: bigint]
usersBelowMinimum: Long = 0L

### Inspect source timestamps

MovieLens stores Unix timestamps in seconds. The parser requires `Long` values; this additional check rejects null and nonpositive timestamps for this post-1970 dataset. The minimum and maximum timestamps are converted to UTC dates for inspection. These basic checks do not verify the correctness of every recorded date.

The aggregation produces one summary row. `first()` retrieves that row, `getAs[Long]` reads a named field, and `Instant.ofEpochSecond` converts it to a UTC instant. The original timestamp column is unchanged. Timestamps are not ALS features, and this inspection does not define a temporal split.


In [12]:
import java.time.Instant

val invalidTimestampCount = ratingsDF.filter(
  col("timestamp").isNull || col("timestamp") <= 0L
).count()

println(s"Records with missing or nonpositive timestamps: $invalidTimestampCount")
assert(invalidTimestampCount == 0L, "MovieLens timestamps must be present and positive Unix seconds.")

val timestampBounds = ratingsDF.agg(
  min("timestamp").as("earliestTimestamp"),
  max("timestamp").as("latestTimestamp")
).first()

val earliestRating = Instant.ofEpochSecond(timestampBounds.getAs[Long]("earliestTimestamp"))
val latestRating = Instant.ofEpochSecond(timestampBounds.getAs[Long]("latestTimestamp"))

println(s"Earliest rating (UTC): $earliestRating")
println(s"Latest rating (UTC): $latestRating")


Records with missing or nonpositive timestamps: 0


Earliest rating (UTC): 2000-04-25T23:05:32Z


Latest rating (UTC): 2003-02-28T17:49:50Z


import java.time.Instant
invalidTimestampCount: Long = 0L
timestampBounds: org.apache.spark.sql.Row = [956703932,1046454590]
earliestRating: Instant = 2000-04-25T23:05:32Z
latestRating: Instant = 2003-02-28T17:49:50Z

## 4. ALS baseline

**Status:** Not implemented. This section contains no model-training code or fitted-model results.


## 5. Baseline recommendations

**Status:** Not implemented. No candidate rankings or recommendation lists have been generated.


## 6. Stop the Spark session

`spark.stop()` shuts down the Spark session and its underlying Spark context, releasing their resources. Run this cell after data inspection and any subsequent Spark work. Further Spark operations require initializing a session again.


In [13]:
spark.stop()
println("Spark session stopped.")


Spark session stopped.
